In [1]:
from chggen.common.sample_utils import CSP_Generator
from chggen.common.data_utils import mkdir
from chggen.common.sample_utils import get_inpaint_data_fromHost
from chggen.common.sample_utils import get_batch_inpaint_data_fromHost
from chggen.common.sample_utils import get_coarse_grain_framework, filter_nan_structure, compute_ewald_energy_single_structure

from types import SimpleNamespace
import numpy as np

from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from pymatgen.io.ase import AseAtomsAdaptor
from pymatgen.core import Structure, Composition, Element, Lattice


import pandas as pd

import time
from datetime import datetime

/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
csp = CSP_Generator(chggen_path = "./files/cut_7_conv_3_epoch=27-val_loss=0.87.ckpt",
                    device='cuda:6')

/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/torch/jit/_check.py:172: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn("The TorchScript type system doesn't support "


CHGNet initialized with 412,525 parameters
CHGNet will run on cuda:6


In [3]:
ld_kwargs = SimpleNamespace(
        n_step_each = 5,            # Corrector
        min_sigma = 0.01,
        num_noise_level = 200,
        signal_to_noise_ratio = 0.4,
        save_traj = False,
        disable_bar = False,
    )

In [4]:
gen_kwargs = SimpleNamespace(
        num_gen = 3, # number of structures generated from the cubic lattice
        num_mutation = 2, # number of mutations during the relax-generation iteration
        num_cell = 1, # number of times to the formula
        ehull_cutoff = 0.06,
        )
                                           

In [5]:
chemical_formula = 'MgSP2S5'
atomic_volume = 24
#  Generate seven different bravis lattices via diffusion
s_list_Bravis = csp.generate_structures_from_Bravis(comp_str= chemical_formula, atom_volume= atomic_volume,
                                                    gen_kwargs=gen_kwargs, ld_kwargs=ld_kwargs, )


s_list_Bravis = filter_nan_structure(s_list_Bravis)

num_atoms [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9]
num_atoms tensor([9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9,
        9, 9, 9, 9], device='cuda:6')
[(6.000000000000001, 6.000000000000001, 6.000000000000001), (6.000000000000001, 6.000000000000001, 6.000000000000001), (6.000000000000001, 6.000000000000001, 6.000000000000001), (6.000000000000001, 6.000000000000001, 6.000000000000001), (6.6038544977892535, 6.6038544977892535, 4.95289087334194), (6.273095502896523, 6.273095502896523, 5.488958565034458), (5.768998281229633, 5.768998281229633, 6.490123066383338), (5.569906600335335, 5.569906600335335, 6.962383250419169), (6.6038544977892535, 4.95289087334194, 6.6038544977892535), (6.273095502896523, 5.488958565034458, 6.273095502896523), (5.768998281229633, 6.490123066383338, 5.768998281229633), (5.569906600335335, 6.962383250419169, 5.569906600335335), (6.639455479471332, 4.979591609603499, 6.639455479471332), (6.3069

100%|██████████| 199/199 [00:26<00:00,  7.38it/s]


In [6]:
s_list_relax = []
for s in s_list_Bravis:
    atoms = AseAtomsAdaptor().get_atoms(s)

    result = csp.relaxer.relax(atoms= atoms,
                        fmax = 0.1,
                        steps = 2000,
                        relax_cell = True,
                        verbose = True,
                        # trajectory_path = None,
    )
    s_relax = result["final_structure"]
    s_list_relax.append(s_relax)

      Step     Time          Energy         fmax
*Force-consistent energies used in optimization.
FIRE:    0 16:23:27      -42.980649*       7.1097
FIRE:    1 16:23:27      -42.572562*      12.9753
FIRE:    2 16:23:27      -43.153211*       6.2840
FIRE:    3 16:23:27      -43.129346*       8.9492
FIRE:    4 16:23:27      -43.209713*       6.5840
FIRE:    5 16:23:27      -43.302192*       3.2351
FIRE:    6 16:23:27      -43.340592*       1.3345
FIRE:    7 16:23:27      -43.323761*       3.6447
FIRE:    8 16:23:27      -43.327744*       3.4898
FIRE:    9 16:23:27      -43.335134*       3.1687
FIRE:   10 16:23:27      -43.344832*       2.6795
FIRE:   11 16:23:27      -43.355480*       2.1259
FIRE:   12 16:23:27      -43.365895*       1.4997
FIRE:   13 16:23:27      -43.374998*       1.3090
FIRE:   14 16:23:28      -43.382212*       1.2678
FIRE:   15 16:23:28      -43.388254*       1.2087
FIRE:   16 16:23:28      -43.393949*       1.4011
FIRE:   17 16:23:28      -43.401352*       1.8242
FI

In [7]:
# len(1.25)

In [8]:
host_structure_list = []
num_intercalat_list = []

for s in s_list_relax:
    analyzer_asGen = SpacegroupAnalyzer(structure= s, symprec= 0.15, angle_tolerance= 15)
    symbol_asGen = analyzer_asGen.get_space_group_symbol()
    print("As generated spacegroup: ", symbol_asGen)

    s_frame, symbol_frame, num_species = get_coarse_grain_framework(s, species_to_remove= 'Li')
    s_frame = s_frame.get_primitive_structure()

    if symbol_frame in ['P1', 'P-1', 'Pm']: # or symbol_inpaint== 'P-1' or symbol_inpaint == 'Pm':
        continue

    host_structure_list.append(s_frame)
    num_intercalat_list.append(int(num_species))
    print("--"*10)

As generated spacegroup:  P1
CG spacegroup:  Cm
--------------------
As generated spacegroup:  P1
CG spacegroup:  C2
--------------------
As generated spacegroup:  P1
CG spacegroup:  P1
As generated spacegroup:  P1
CG spacegroup:  Pm
As generated spacegroup:  P1
CG spacegroup:  P1
As generated spacegroup:  P1
CG spacegroup:  P1
As generated spacegroup:  P1
CG spacegroup:  Cm
--------------------
As generated spacegroup:  P1
CG spacegroup:  P1
As generated spacegroup:  P1
CG spacegroup:  Pm
As generated spacegroup:  P1
CG spacegroup:  Pm
As generated spacegroup:  P1
CG spacegroup:  Cm
--------------------
As generated spacegroup:  P1
CG spacegroup:  Cm
--------------------
As generated spacegroup:  P1
CG spacegroup:  P1
As generated spacegroup:  P1
CG spacegroup:  P1
As generated spacegroup:  P1
CG spacegroup:  C2
--------------------
As generated spacegroup:  P1
CG spacegroup:  P1
As generated spacegroup:  P1
CG spacegroup:  P1
As generated spacegroup:  P1
CG spacegroup:  Pm
As generat

In [9]:
len(host_structure_list)

11

In [10]:
s_list_inpaint = csp.generate_from_host_structure(host_structure_list= host_structure_list * 3,
                                 num_intercalant_list= num_intercalat_list * 3,
                                 ld_kwargs=ld_kwargs, 
                                 species= 'Li')

/home/zhongpc/chggen/cond_gen/chggen/common/sample_utils.py:440: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  cur_frac_coords = torch.tensor(cur_frac_coords, dtype = torch.float32)
100%|██████████| 959/959 [01:24<00:00, 11.40it/s]


In [11]:
s_list_inpaint_conventional_unit = []

for s in s_list_inpaint:
    analyzer_inpaint = SpacegroupAnalyzer(structure= s, symprec= 0.2, angle_tolerance= 15)
    symbol_inpaint = analyzer_inpaint.get_space_group_symbol()
    

    

    if symbol_inpaint== 'P1' or symbol_inpaint== 'P-1':
        continue
    else:
        print(symbol_inpaint)
        s_inpaint_conventional_unit = analyzer_inpaint.get_conventional_standard_structure()
        s_list_inpaint_conventional_unit.append(s_inpaint_conventional_unit)

Cm
C2
Cm
Cm
Cm
C2
C2
Cm
C2
C2
C2
Cm
C2
Cm
Cm
Cm
C2
C2
Cm
C2
C2
C2
Cm
C2
Cm
Cm
Cm
C2
C2
Cm
C2
C2
C2


In [12]:
E0_atom_list = []
for ii, s in enumerate(s_list_inpaint_conventional_unit):
    Ewald_per_atom = compute_ewald_energy_single_structure(s) / s.num_sites
    
    
    prediction = csp.chgnet.predict_structure(s)
    E0_atom = prediction['e']
    F_max = np.max(np.abs(prediction['f']))

    E0_atom_list.append(-E0_atom)
    # print("--"*10)
    # print(ii)
    # print(Ewald_per_atom, s.composition)

    # print("E0_per_atom: ", E0_atom, "F_max: ", F_max)

    # print("--"*10)

In [13]:
# Calculate the threshold value
threshold = np.percentile(E0_atom_list, 50)

# Filter out the structures with E0_atom values above the threshold
filtered_structures = [structure for structure, e0_atom in zip(s_list_inpaint_conventional_unit, E0_atom_list) if e0_atom > threshold]


In [14]:
np.sort(E0_atom_list)

array([3.642997 , 3.657957 , 3.668755 , 4.3700705, 4.3717275, 4.3732367,
       4.4753723, 4.475958 , 4.4779043, 4.5420794, 4.5424037, 4.5449553,
       4.5555058, 4.5560803, 4.556115 , 4.556505 , 4.5571795, 4.559849 ,
       4.620657 , 4.625559 , 4.630941 , 4.730168 , 4.730771 , 4.732374 ,
       4.7749777, 4.776324 , 4.777913 , 4.8273025, 4.8275213, 4.8283076,
       4.887816 , 4.8879347, 4.8880854], dtype=float32)

In [15]:
threshold

4.5571794509887695

In [16]:
filtered_structures

[Structure Summary
 Lattice
     abc : 6.333327030454292 13.55051109868575 5.741742322061592
  angles : 90.0 92.5847198938527 90.0
  volume : 492.25396889521085
       A : 0.0 6.333327030454292 0.0
       B : 13.55051109868575 0.0 0.0
       C : 0.0 -0.2589329011641028 -5.735900857376109
     pbc : True True True
 PeriodicSite: Mg (9.321, 3.167, 0.0) [0.5, 0.6879, 0.0]
 PeriodicSite: Mg (2.546, 0.0, 0.0) [0.0, 0.1879, 0.0]
 PeriodicSite: P (11.18, 3.037, -2.868) [0.5, 0.8254, 0.5]
 PeriodicSite: P (13.33, 3.167, 0.0) [0.5, 0.9841, 0.0]
 PeriodicSite: P (4.409, -0.1295, -2.868) [0.0, 0.3254, 0.5]
 PeriodicSite: P (6.56, 0.0, 0.0) [0.0, 0.4841, 0.0]
 PeriodicSite: S (7.45, 4.541, -4.733) [0.7508, 0.5498, 0.8251]
 PeriodicSite: S (11.63, 3.964, -4.845) [0.6605, 0.858, 0.8448]
 PeriodicSite: S (9.892, 4.267, -2.094) [0.6887, 0.73, 0.365]
 PeriodicSite: S (11.63, 2.11, -0.8905) [0.3395, 0.858, 0.1552]
 PeriodicSite: S (7.45, 1.533, -1.003) [0.2492, 0.5498, 0.1749]
 PeriodicSite: S (9.892, 1

In [17]:
ROOT ='files/inpaint_'+chemical_formula+'_LPS' 
mkdir(ROOT)

for ii, s in enumerate(filtered_structures):
    s.to(filename=ROOT +'/_'+str(ii)+'.cif')